# Phase 16 — Final Evidence Consolidation

CPU-only deterministic reporting over twelve frozen result ZIPs. Phase 14 is primary locked-test evidence; Phases 15A/15B are separate exploratory external tasks. No model, raw dataset, training, tuning, routing, inference, or test reopening occurs.

In [1]:
"""Pure-Python integrity and reporting helpers for MentalDistress Phase 16."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path, PurePosixPath
import csv
import hashlib
import io
import json
import os
import zipfile

import pandas as pd


FIXED_ZIP_TIME = (2020, 1, 1, 0, 0, 0)


@dataclass(frozen=True)
class ArchiveSpec:
    phase_id: str
    filename: str
    sha256: str
    root: str
    manifest_member: str | None
    required_members: tuple[str, ...]
    evidence_tier: str
    data_scope: str


@dataclass(frozen=True)
class VerifiedArchive:
    spec: ArchiveSpec
    path: Path
    size_bytes: int
    archive_sha256: str
    member_names: tuple[str, ...]
    manifest_sha256: str | None


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_member_names(names: list[str]) -> tuple[str, ...]:
    seen: set[str] = set()
    files: set[str] = set()
    directories: set[str] = set()
    roots: set[str] = set()
    for raw in names:
        if not raw or "\\" in raw or raw.startswith("/"):
            raise ValueError(f"unsafe ZIP member: {raw!r}")
        path = PurePosixPath(raw)
        if path.is_absolute() or any(part in ("", ".", "..") for part in path.parts):
            raise ValueError(f"unsafe ZIP member: {raw!r}")
        canonical = path.as_posix()
        if canonical in seen:
            raise ValueError(f"duplicate ZIP member: {canonical}")
        seen.add(canonical)
        roots.add(path.parts[0])
        if raw.endswith("/"):
            directories.add(canonical.rstrip("/"))
        else:
            files.add(canonical)
        for parent in path.parents:
            if parent.as_posix() != ".":
                directories.add(parent.as_posix())
    conflicts = files & directories
    if conflicts:
        raise ValueError(f"file/directory ZIP path conflict: {sorted(conflicts)}")
    if len(roots) != 1:
        raise ValueError(f"ZIP must contain exactly one root: {sorted(roots)}")
    return tuple(sorted(seen))


def verify_manifest_rows(
    archive: zipfile.ZipFile, *, root: str, manifest_member: str
) -> tuple[int, str]:
    manifest_name = f"{root}/{manifest_member}"
    payload = archive.read(manifest_name)
    rows = list(csv.DictReader(io.StringIO(payload.decode("utf-8-sig"))))
    if not rows or set(rows[0]) != {"path", "sha256", "bytes"}:
        raise ValueError(f"invalid manifest columns: {manifest_name}")
    relative_files = {
        name[len(root) + 1 :]
        for name in archive.namelist()
        if not name.endswith("/") and name.startswith(root + "/") and name != manifest_name
    }
    paths = [row["path"] for row in rows]
    if len(paths) != len(set(paths)):
        raise ValueError(f"duplicate manifest path: {manifest_name}")
    if set(paths) != relative_files:
        missing = sorted(relative_files - set(paths))
        extra = sorted(set(paths) - relative_files)
        raise ValueError(f"manifest membership mismatch; missing={missing}, extra={extra}")
    for row in rows:
        rel = row["path"]
        validate_member_names([f"{root}/{rel}"])
        member = f"{root}/{rel}"
        content = archive.read(member)
        if int(row["bytes"]) != len(content):
            raise ValueError(f"manifest byte mismatch: {rel}")
        if row["sha256"] != sha256_bytes(content):
            raise ValueError(f"manifest SHA-256 mismatch: {rel}")
    return len(rows), sha256_bytes(payload)


def verify_archive(path: Path, spec: ArchiveSpec) -> VerifiedArchive:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"required archive not found: {path}")
    observed = sha256_file(path)
    if observed != spec.sha256:
        raise ValueError(f"archive SHA-256 mismatch for {path.name}: {observed}")
    with zipfile.ZipFile(path) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise ValueError(f"ZIP CRC failure: {bad_member}")
        names = validate_member_names(archive.namelist())
        observed_roots = {PurePosixPath(name).parts[0] for name in names}
        if observed_roots != {spec.root}:
            raise ValueError(f"unexpected archive root: {sorted(observed_roots)}")
        files = {name for name in names if not name.endswith("/")}
        required = {f"{spec.root}/{name}" for name in spec.required_members}
        missing = sorted(required - files)
        if missing:
            raise ValueError(f"missing required members: {missing}")
        manifest_sha = None
        if spec.manifest_member:
            _, manifest_sha = verify_manifest_rows(
                archive, root=spec.root, manifest_member=spec.manifest_member
            )
    return VerifiedArchive(
        spec=spec,
        path=path.resolve(),
        size_bytes=path.stat().st_size,
        archive_sha256=observed,
        member_names=names,
        manifest_sha256=manifest_sha,
    )


def read_bytes_member(verified: VerifiedArchive, relative_member: str) -> bytes:
    validate_member_names([f"{verified.spec.root}/{relative_member}"])
    full_name = f"{verified.spec.root}/{relative_member}"
    if full_name not in verified.member_names:
        raise FileNotFoundError(f"archive member not found: {full_name}")
    with zipfile.ZipFile(verified.path) as archive:
        return archive.read(full_name)


def read_csv_member(verified: VerifiedArchive, relative_member: str) -> pd.DataFrame:
    return pd.read_csv(io.BytesIO(read_bytes_member(verified, relative_member)))


def read_json_member(verified: VerifiedArchive, relative_member: str) -> dict:
    payload = json.loads(read_bytes_member(verified, relative_member).decode("utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"JSON member must contain an object: {relative_member}")
    return payload


def write_json(path: Path, payload: dict) -> None:
    Path(path).write_text(
        json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def write_frame(path: Path, frame: pd.DataFrame) -> None:
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.12g")


def write_sha256_manifest(root: Path) -> pd.DataFrame:
    root = Path(root)
    manifest_path = root / "sha256_manifest.csv"
    rows = [
        {"path": path.relative_to(root).as_posix(), "sha256": sha256_file(path), "bytes": path.stat().st_size}
        for path in sorted(root.rglob("*"), key=lambda item: item.as_posix())
        if path.is_file() and path != manifest_path
    ]
    frame = pd.DataFrame(rows, columns=["path", "sha256", "bytes"])
    write_frame(manifest_path, frame)
    return frame


def verify_output_manifest(root: Path) -> pd.DataFrame:
    root = Path(root)
    manifest = pd.read_csv(root / "sha256_manifest.csv", dtype={"path": str, "sha256": str})
    if list(manifest.columns) != ["path", "sha256", "bytes"]:
        raise ValueError("invalid output manifest columns")
    if manifest["path"].duplicated().any():
        raise ValueError("duplicate output manifest path")
    actual = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path.name != "sha256_manifest.csv"
    }
    if set(manifest["path"]) != actual:
        raise ValueError("output manifest membership mismatch")
    for row in manifest.itertuples(index=False):
        path = root / row.path
        if path.stat().st_size != int(row.bytes) or sha256_file(path) != row.sha256:
            raise ValueError(f"output manifest mismatch: {row.path}")
    return manifest


def deterministic_zip(source_dir: Path, archive_path: Path) -> str:
    source_dir = Path(source_dir)
    archive_path = Path(archive_path)
    with zipfile.ZipFile(
        archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9
    ) as archive:
        for path in sorted(source_dir.rglob("*"), key=lambda item: item.as_posix()):
            if not path.is_file():
                continue
            name = f"{source_dir.name}/{path.relative_to(source_dir).as_posix()}"
            info = zipfile.ZipInfo(name, FIXED_ZIP_TIME)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.create_system = 0
            info.external_attr = 0
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)
    return sha256_file(archive_path)


def assert_no_raw_text_columns(frame: pd.DataFrame) -> None:
    prohibited = ("text", "selftext", "content", "sentence", "post_body")
    bad = [column for column in frame.columns if any(token in column.lower() for token in prohibited)]
    if bad:
        raise ValueError(f"raw-text-like columns are prohibited: {bad}")


ARCHIVE_ORDER = (
    "step2_3", "phase3_4", "phase5", "phase6", "phase8", "phase9",
    "phase10", "phase11", "phase12", "phase14", "phase15a", "phase15b",
)
HARD_PAIRS = ("Frustrated|Others", "Suicidal|Depressed", "Depressed|Others")


def _spec(
    phase_id: str,
    filename: str,
    digest: str,
    root: str,
    required: tuple[str, ...],
    tier: str,
    scope: str,
    *,
    manifest: bool = True,
) -> ArchiveSpec:
    return ArchiveSpec(
        phase_id, filename, digest, root,
        "sha256_manifest.csv" if manifest else None,
        required, tier, scope,
    )


ARCHIVE_SPECS = {
    "step2_3": _spec(
        "step2_3", "MentalDistress_Step2_Step3_Artifacts.zip",
        "ae48bbef0a6796329ded207f9a15eca9988dabfae3cd80ecec64a3b201855415",
        "mental_distress_step2_step3_artifacts",
        ("run_manifest.json", "step3_validation_summary.csv", "tfidf_logistic_regression_validation_metrics.json", "tfidf_linear_svm_validation_metrics.json", "dataset_integrity_report.json"),
        "PROVENANCE_ONLY", "development", manifest=False,
    ),
    "phase3_4": _spec(
        "phase3_4", "mentalroberta_phase3_freeze_phase4_boundary_analysis_RESULTS.zip",
        "ad1611cb441efbc33f7d9a5f1e35975aa0782f867b4be6edcd9104b2673542c3",
        "phase4_boundary_analysis_results",
        ("phase3_phase4_execution_summary.json", "phase3_freeze/phase3_aggregate_metrics.csv", "phase3_freeze/phase3_freeze_manifest.json", "phase4/phase4_pair_stability_summary.csv", "phase4/phase4_rank_agreement.json"),
        "PROVENANCE_ONLY", "development",
    ),
    "phase5": _spec(
        "phase5", "MentalDistress_Phase5_Bootstrap_Stability_Results.zip",
        "9b8ad4a04ff075ac1eb6a604401c2546a1165f3c038a9ae8eefb6af4f8f6bab5",
        "phase5_bootstrap_stability_results",
        ("phase5_execution_summary.json", "phase5_hard_pair_selection.json", "phase5_pair_pooled_summary.csv"),
        "PROVENANCE_ONLY", "development",
    ),
    "phase6": _spec(
        "phase6", "MentalDistress_Phase6_Others_Sensitivity_Results.zip",
        "33b11811899950354db6e3173913d4f1633943a193752c543c4a661ea02db10a",
        "phase6_others_sensitivity_results",
        ("phase6_execution_summary.json", "phase6_frozen_h_sensitivity.json", "phase6_original_named_pair_summary.csv", "phase6_bootstrap_pair_summary.csv", "phase6_error_concentration_summary.csv"),
        "SECONDARY_INTERNAL", "development",
    ),
    "phase8": _spec(
        "phase8", "MentalDistress_Phase8_All_Pair_Specialists_Results.zip",
        "6d30115f9da6a1963b2fc74eb1b4c057ecb685ca7ae6dcf0c61d177bb3fa89e1",
        "phase8_all_pair_specialists_results",
        ("phase8_execution_summary.json", "phase8_pair_summary.csv", "phase8_seed_summary.csv", "phase8_protocol.json"),
        "PROVENANCE_ONLY", "development",
    ),
    "phase9": _spec(
        "phase9", "MentalDistress_Phase9_Routing_Calibration_Results.zip",
        "9db43d16d0acd7c0fb68129ee2c2e72b88ed3a4ff73c8df7a7ecfd80fcc68ba3",
        "phase9_routing_calibration_results",
        ("phase9_selected_config.json", "phase9_routing_grid.csv", "phase9_routing_seedwise.csv"),
        "PROVENANCE_ONLY", "development",
    ),
    "phase10": _spec(
        "phase10", "MentalDistress_Phase10_Controls_and_Allocation_Results.zip",
        "7d9e0065632e024d4f9aece380a72cefec2b2bc3e03e18bf51e076229a421ff3",
        "phase10_controls_allocation_results",
        ("phase10_execution_summary.json", "phase10_control_summary.csv", "phase10_selected_allocation_rank.json", "phase10_threshold_sensitivity.csv", "phase10_protocol.json"),
        "SECONDARY_INTERNAL", "development",
    ),
    "phase11": _spec(
        "phase11", "MentalDistress_Phase11_Mechanism_Reliability_Robustness_Results.zip",
        "f3a0ae1499d8cee01e370cadcf3b09ee3ea36bbd9f0eb3e0bd0428fe9abc1068",
        "phase11_mechanism_reliability_robustness_results",
        ("phase11_execution_summary.json", "phase11_original_metric_summary.csv", "phase11_bcr_summary.csv", "phase11_flat_reliability_summary.csv", "phase11_aurc_summary.csv", "phase11_lexical_robustness_summary.csv", "phase11_suicidal_summary.csv", "phase11_phase10_reproduction.csv"),
        "SECONDARY_INTERNAL", "development",
    ),
    "phase12": _spec(
        "phase12", "MentalDistress_Phase12_Efficiency_Benchmarking_Results.zip",
        "c0ed1d6af54ae0abf377c818fc44a66ff150eaf4e087dd0511ae527bca24adac",
        "phase12_efficiency_benchmarking_results",
        ("phase12_execution_summary.json", "phase12_efficiency_consolidated_summary.csv", "phase12_parameter_counts_summary.csv", "phase12_performance_activation_summary.csv", "phase12_vram_summary.csv", "phase12_gpu_metadata.csv", "phase12_structure_audit.csv"),
        "SECONDARY_INTERNAL", "development",
    ),
    "phase14": _spec(
        "phase14", "MentalDistress_Phase14_Final_Locked_Test_Results.zip",
        "4f83c651891650f2f8f9d54dda9088f0e0caf19ef4fb73de91ed1e9893e32e8b",
        "phase14_final_locked_test_results",
        ("phase14_execution_summary.json", "phase14_test_opening_record.json", "phase14_equal_seed_summary.csv", "phase14_paired_bootstrap.csv", "phase14_mcnemar_holm.csv", "phase14_routing_seedwise.csv", "phase14_lexical_robustness_summary.csv", "phase14_suicidal_false_negatives.csv", "phase14_claim_guardrails.csv", "phase14_structure_audit.csv"),
        "PRIMARY_LOCKED_TEST", "locked_test",
    ),
    "phase15a": _spec(
        "phase15a", "MentalDistress_Phase15A_SDCNL_External_Validation_Results.zip",
        "8fcb24274ae71cca3af1ecd37f700afbcc7a9b32cb2c05587edb989e38aa7b66",
        "phase15a_sdcnl_external_validation_results",
        ("phase15_execution_summary.json", "external_equal_seed_summary.csv", "external_paired_bootstrap.csv", "external_mcnemar_holm.csv", "external_label_space_shift.csv", "external_claim_guardrails.csv", "phase15_structure_audit.csv"),
        "EXPLORATORY_EXTERNAL", "external_exploratory",
    ),
    "phase15b": _spec(
        "phase15b", "MentalDistress_Phase15B_DepressionReddit_External_Validation_Results.zip",
        "9eb88b9134ed903227d01bebcc10d74aa7c79bbd5dd7198b19f39c4c6069a395",
        "phase15b_depressionreddit_external_validation_results",
        ("phase15_execution_summary.json", "external_equal_seed_summary.csv", "external_paired_bootstrap.csv", "external_mcnemar_holm.csv", "external_label_space_shift.csv", "external_claim_guardrails.csv", "phase15_structure_audit.csv"),
        "EXPLORATORY_EXTERNAL", "external_exploratory",
    ),
}


def resolve_required_archives(workspace: Path) -> dict[str, VerifiedArchive]:
    workspace = Path(workspace).resolve()
    resolved: dict[str, VerifiedArchive] = {}
    for phase_id in ARCHIVE_ORDER:
        spec = ARCHIVE_SPECS[phase_id]
        path = workspace / spec.filename
        resolved[phase_id] = verify_archive(path, spec)
    return resolved


def _gate(rows: list[dict], phase: str, check: str, observed, expected, source: str) -> None:
    passed = observed == expected
    rows.append({
        "phase_id": phase,
        "check": check,
        "observed": json.dumps(observed, sort_keys=True) if isinstance(observed, (dict, list, tuple)) else observed,
        "expected": json.dumps(expected, sort_keys=True) if isinstance(expected, (dict, list, tuple)) else expected,
        "passed": bool(passed),
        "source_member": source,
    })


def _audit_all_pass(archive: VerifiedArchive, member: str) -> bool:
    frame = read_csv_member(archive, member)
    candidates = [column for column in frame.columns if column.lower() in {"pass", "passed", "status"}]
    if not candidates:
        raise ValueError(f"no pass/status column in {member}")
    values = frame[candidates[0]].astype(str).str.upper()
    return values.isin({"TRUE", "PASS", "1"}).all()


def verify_phase_gate(phase_id: str, archive: VerifiedArchive) -> list[dict]:
    rows: list[dict] = []
    if phase_id == "step2_3":
        source = "run_manifest.json"
        data = read_json_member(archive, source)
        _gate(rows, phase_id, "status", data.get("status"), "PASS", source)
        _gate(rows, phase_id, "test_evaluation", data.get("test_evaluation"), "NOT_PERFORMED", source)
        _gate(rows, phase_id, "train_sha256", data["input_sha256"].get("train"), "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b", source)
        _gate(rows, phase_id, "validation_sha256", data["input_sha256"].get("validation"), "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165", source)
        _gate(rows, phase_id, "test_sha256", data["input_sha256"].get("test"), "e3a61197d1d4a57e61269be260570cc5aa5b504275f0370b1c379f0453517270", source)
        for key in ("saved_artifact_reload_exact_predictions", "second_refit_exact_predictions"):
            _gate(rows, phase_id, key, data["reproducibility"].get(key), True, source)
    elif phase_id == "phase3_4":
        source = "phase3_phase4_execution_summary.json"
        data = read_json_member(archive, source)
        for key, expected in (("status", "PASS"), ("completed_seeds", 10), ("test_accessed", False), ("phase5_required", True), ("final_hard_pairs_frozen", False)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
    elif phase_id == "phase5":
        source = "phase5_hard_pair_selection.json"
        data = read_json_member(archive, source)
        _gate(rows, phase_id, "status", data.get("status"), "FROZEN", source)
        _gate(rows, phase_id, "K", data.get("K"), 3, source)
        _gate(rows, phase_id, "H", data.get("H"), list(HARD_PAIRS), source)
    elif phase_id == "phase6":
        source = "phase6_execution_summary.json"
        data = read_json_member(archive, source)
        _gate(rows, phase_id, "status", data.get("status"), "PASS", source)
        _gate(rows, phase_id, "frozen_h", data.get("frozen_h"), list(HARD_PAIRS), source)
        _gate(rows, phase_id, "frozen_h_unchanged", data.get("frozen_h_unchanged"), True, source)
    elif phase_id == "phase8":
        source = "phase8_execution_summary.json"
        data = read_json_member(archive, source)
        for key, expected in (("status", "PASS"), ("model_seed_count", 10), ("pair_count", 10), ("head_count", 100), ("test_accessed", False)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
    elif phase_id == "phase9":
        source = "phase9_selected_config.json"
        data = read_json_member(archive, source)
        for key, expected in (("H", list(HARD_PAIRS)), ("K_fixed", 3), ("selected_tau", 0.1), ("test_accessed", False)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
    elif phase_id == "phase10":
        source = "phase10_execution_summary.json"
        data = read_json_member(archive, source)
        for key, expected in (("status", "PASS"), ("phase9_reproduction_pass_count", 10), ("frozen_tau", 0.1), ("seed_count", 10), ("test_accessed", False)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
    elif phase_id in {"phase11", "phase12"}:
        number = phase_id.removeprefix("phase")
        source = f"phase{number}_execution_summary.json"
        data = read_json_member(archive, source)
        for key, expected in (("status", "PASS"), ("phase10_reproduction_pass_count", 10), ("seed_count", 10), ("test_accessed", False)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
        if phase_id == "phase12":
            _gate(rows, phase_id, "structure_audit", _audit_all_pass(archive, "phase12_structure_audit.csv"), True, "phase12_structure_audit.csv")
    elif phase_id == "phase14":
        source = "phase14_execution_summary.json"
        data = read_json_member(archive, source)
        for key, expected in (("status", "PASS"), ("seed_count", 10), ("test_open_count", 1), ("test_rows", 1491), ("test_accessed", True)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
        opening = read_json_member(archive, "phase14_test_opening_record.json")
        _gate(rows, phase_id, "test_sha256", opening.get("test_sha256"), "e3a61197d1d4a57e61269be260570cc5aa5b504275f0370b1c379f0453517270", "phase14_test_opening_record.json")
        _gate(rows, phase_id, "structure_audit", _audit_all_pass(archive, "phase14_structure_audit.csv"), True, "phase14_structure_audit.csv")
    elif phase_id in {"phase15a", "phase15b"}:
        source = "phase15_execution_summary.json"
        data = read_json_member(archive, source)
        for key, expected in (("status", "PASS"), ("seed_count", 10), ("validation_reproduction_pass_count", 10), ("external_training", False), ("tuning_performed", False), ("routing_performed", False)):
            _gate(rows, phase_id, key, data.get(key), expected, source)
        _gate(rows, phase_id, "structure_audit", _audit_all_pass(archive, "phase15_structure_audit.csv"), True, "phase15_structure_audit.csv")
    else:
        raise KeyError(f"unknown phase gate: {phase_id}")
    return rows


def archive_inventory_frame(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows = [{
        "phase_id": phase_id,
        "filename": item.spec.filename,
        "size_bytes": item.size_bytes,
        "archive_sha256": item.archive_sha256,
        "root": item.spec.root,
        "manifest_member": item.spec.manifest_member or "",
        "manifest_sha256": item.manifest_sha256 or "",
        "member_count": len([name for name in item.member_names if not name.endswith("/")]),
        "evidence_tier": item.spec.evidence_tier,
        "data_scope": item.spec.data_scope,
        "status": "PASS",
    } for phase_id, item in archives.items()]
    return pd.DataFrame(rows)


def verify_all_phase_gates(
    archives: dict[str, VerifiedArchive],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = [row for phase_id in ARCHIVE_ORDER for row in verify_phase_gate(phase_id, archives[phase_id])]
    gates = pd.DataFrame(rows)
    if not gates["passed"].all():
        failed = gates.loc[~gates["passed"]].to_dict("records")
        raise ValueError(f"upstream phase gate failure: {failed}")
    return archive_inventory_frame(archives), gates


PROVENANCE_COLUMNS = [
    "source_phase", "source_archive", "source_member", "evidence_tier",
    "data_scope", "system_id", "model_seed", "metric", "value",
    "aggregation_unit", "limitation", "comparability_group",
]

EVIDENCE_MEMBERS = {
    "step2_3": ("step3_validation_summary.csv", "run_manifest.json"),
    "phase3_4": ("phase3_freeze/phase3_aggregate_metrics.csv", "phase4/phase4_pair_stability_summary.csv", "phase4/phase4_rank_agreement.json"),
    "phase5": ("phase5_hard_pair_selection.json", "phase5_pair_pooled_summary.csv"),
    "phase6": ("phase6_original_named_pair_summary.csv", "phase6_bootstrap_pair_summary.csv", "phase6_error_concentration_summary.csv"),
    "phase8": ("phase8_pair_summary.csv", "phase8_seed_summary.csv"),
    "phase9": ("phase9_selected_config.json", "phase9_routing_grid.csv"),
    "phase10": ("phase10_control_summary.csv", "phase10_selected_allocation_rank.json", "phase10_threshold_sensitivity.csv"),
    "phase11": ("phase11_original_metric_summary.csv", "phase11_bcr_summary.csv", "phase11_flat_reliability_summary.csv", "phase11_aurc_summary.csv", "phase11_lexical_robustness_summary.csv", "phase11_suicidal_summary.csv"),
    "phase12": ("phase12_efficiency_consolidated_summary.csv", "phase12_parameter_counts_summary.csv", "phase12_performance_activation_summary.csv", "phase12_vram_summary.csv", "phase12_gpu_metadata.csv"),
    "phase14": ("phase14_equal_seed_summary.csv", "phase14_paired_bootstrap.csv", "phase14_mcnemar_holm.csv", "phase14_routing_seedwise.csv", "phase14_lexical_robustness_summary.csv", "phase14_suicidal_false_negatives.csv", "phase14_claim_guardrails.csv"),
    "phase15a": ("external_equal_seed_summary.csv", "external_paired_bootstrap.csv", "external_mcnemar_holm.csv", "external_label_space_shift.csv", "external_claim_guardrails.csv"),
    "phase15b": ("external_equal_seed_summary.csv", "external_paired_bootstrap.csv", "external_mcnemar_holm.csv", "external_label_space_shift.csv", "external_claim_guardrails.csv"),
}


def _row(
    archives: dict[str, VerifiedArchive], phase_id: str, member: str, *,
    system_id: str, metric: str, value, aggregation_unit: str,
    limitation: str, comparability_group: str, model_seed="",
    source_phase: str | None = None, **extra,
) -> dict:
    item = archives[phase_id]
    return {
        "source_phase": source_phase or phase_id.replace("phase", "").replace("step2_3", "2/3"),
        "source_archive": item.spec.filename,
        "source_member": member,
        "evidence_tier": item.spec.evidence_tier,
        "data_scope": item.spec.data_scope,
        "system_id": system_id,
        "model_seed": model_seed,
        "metric": metric,
        "value": value,
        "aggregation_unit": aggregation_unit,
        "limitation": limitation,
        "comparability_group": comparability_group,
        **extra,
    }


def _frame(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    for column in PROVENANCE_COLUMNS:
        if column not in frame:
            frame[column] = ""
    ordered = PROVENANCE_COLUMNS + [column for column in frame.columns if column not in PROVENANCE_COLUMNS]
    result = frame[ordered]
    assert_no_raw_text_columns(result)
    return result


def build_experiment_ledger(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    purposes = {
        "step2_3": "Classical validation baselines",
        "phase3_4": "Ten-seed global freeze and boundary discovery",
        "phase5": "Bootstrap stability and hard-pair freeze",
        "phase6": "Others sensitivity analysis",
        "phase8": "All-pair specialist training",
        "phase9": "Routing threshold calibration",
        "phase10": "Controls and allocation analysis",
        "phase11": "Mechanism, reliability, and robustness",
        "phase12": "Efficiency benchmarking",
        "phase14": "Final locked-test evaluation",
        "phase15a": "SDCNL exploratory boundary transfer",
        "phase15b": "Depression Reddit exploratory proxy transfer",
    }
    rows = []
    for index, phase_id in enumerate(ARCHIVE_ORDER, 1):
        spec = archives[phase_id].spec
        rows.append({
            "order": index, "phase_id": phase_id, "purpose": purposes[phase_id],
            "data_scope": spec.data_scope, "evidence_tier": spec.evidence_tier,
            "source_archive": spec.filename, "source_members": ";".join(EVIDENCE_MEMBERS[phase_id]),
            "frozen_choice": "K=3; H=" + ";".join(HARD_PAIRS) if phase_id in {"phase5", "phase6", "phase8"} else ("tau=0.10" if phase_id in {"phase9", "phase10", "phase11", "phase12", "phase14"} else "none"),
            "claim_scope": "primary internal" if phase_id == "phase14" else ("exploratory external" if phase_id.startswith("phase15") else "development/provenance"),
        })
    return pd.DataFrame(rows)


def build_baseline_boundary_table(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows: list[dict] = []
    legacy = read_csv_member(archives["step2_3"], "step3_validation_summary.csv")
    for record in legacy.to_dict("records"):
        for metric in ("accuracy", "macro_f1", "weighted_f1", "mcc"):
            rows.append(_row(archives, "step2_3", "step3_validation_summary.csv", system_id=record["model"], metric=metric, value=record[metric], aggregation_unit="single validation fit", limitation="Development-only classical baseline.", comparability_group="validation_five_class"))
    global_summary = read_csv_member(archives["phase3_4"], "phase3_freeze/phase3_aggregate_metrics.csv")
    for record in global_summary.to_dict("records"):
        rows.append(_row(archives, "phase3_4", "phase3_freeze/phase3_aggregate_metrics.csv", system_id="MentalRoBERTa_global", metric=record["metric"], value=record["mean"], aggregation_unit="equal mean across 10 model seeds", limitation="Development validation performance; not a locked-test result.", comparability_group="validation_five_class", sample_sd=record["sample_sd"], seed_count=record["seed_count"]))
    boundaries = read_csv_member(archives["phase3_4"], "phase4/phase4_pair_stability_summary.csv")
    for record in boundaries.to_dict("records"):
        rows.append(_row(archives, "phase3_4", "phase4/phase4_pair_stability_summary.csv", system_id=record["pair"], metric="mean_bcr", value=record["mean_bcr"], aggregation_unit="equal mean across 10 model seeds", limitation="Boundary discovery only; H was frozen in Phase 5.", comparability_group="validation_pair_boundary", sample_sd=record["sample_sd_bcr"], top3_frequency=record["top3_frequency"]))
    stable = read_csv_member(archives["phase5"], "phase5_pair_pooled_summary.csv")
    for record in stable.to_dict("records"):
        rows.append(_row(archives, "phase5", "phase5_pair_pooled_summary.csv", system_id=record["pair"], metric="bootstrap_mean_bcr", value=record["pooled_bcr_mean"], aggregation_unit="20,000 within-seed bootstrap observations", limitation="Bootstrap observations are not independent model replications.", comparability_group="validation_pair_bootstrap", ci_low=record["pooled_bcr_ci_lower"], ci_high=record["pooled_bcr_ci_upper"], selected_in_H=record["pair"] in HARD_PAIRS))
    return _frame(rows)


def build_routing_controls_table(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows: list[dict] = []
    grid = read_csv_member(archives["phase9"], "phase9_routing_grid.csv")
    for record in grid.to_dict("records"):
        rows.append(_row(archives, "phase9", "phase9_routing_grid.csv", system_id=f"routing_tau_{record['tau']:.2f}", metric="macro_f1", value=record["mean_macro_f1"], aggregation_unit="equal mean across 10 model seeds", limitation="Validation-only threshold calibration; tau was not selected on test.", comparability_group="validation_five_class", tau=record["tau"], sample_sd=record["sd_macro_f1"], coverage=record["mean_coverage"], selected_tau=record["tau"] == 0.1))
    controls = read_csv_member(archives["phase10"], "phase10_control_summary.csv")
    for record in controls.to_dict("records"):
        rows.append(_row(archives, "phase10", "phase10_control_summary.csv", system_id=record["control_id"], metric="macro_f1", value=record["mean_macro_f1"], aggregation_unit="equal mean across 10 model seeds", limitation="Development control evidence.", comparability_group="validation_five_class", sample_sd=record["sd_macro_f1"], coverage=record["mean_coverage"], delta_vs_A0=record["mean_macro_f1_delta_vs_A0"]))
    return _frame(rows)


def build_mechanism_robustness_table(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows: list[dict] = []
    sources = [
        ("phase11_original_metric_summary.csv", "system_id", "mean_macro_f1", "macro_f1", "sd_macro_f1"),
        ("phase11_flat_reliability_summary.csv", None, "mean", "reliability", "sample_sd"),
        ("phase11_aurc_summary.csv", "system_id", "mean_aurc", "aurc", "sd_aurc"),
        ("phase11_lexical_robustness_summary.csv", "system_id", "mean_macro_f1", "macro_f1", "sd_macro_f1"),
        ("phase11_suicidal_summary.csv", "system_id", "mean_recall", "suicidal_recall", "sd_recall"),
    ]
    for member, system_column, mean_column, metric, sd_column in sources:
        frame = read_csv_member(archives["phase11"], member)
        for record in frame.to_dict("records"):
            system = record.get(system_column, "A0_flat") if system_column else "A0_flat"
            observed_metric = record.get("metric", metric) if metric == "reliability" else metric
            variant = record.get("variant", "original")
            rows.append(_row(archives, "phase11", member, system_id=system, metric=f"{variant}_{observed_metric}", value=record[mean_column], aggregation_unit="equal mean across 10 model seeds", limitation="Observational development analysis; no causal mechanism claim.", comparability_group="validation_five_class", sample_sd=record[sd_column]))
    return _frame(rows)


def build_efficiency_table(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows: list[dict] = []
    summary = read_csv_member(archives["phase12"], "phase12_efficiency_consolidated_summary.csv")
    measures = [
        ("mean_macro_f1", "macro_f1", "sd_macro_f1"),
        ("mean_batch1_median_latency_ms", "batch1_median_latency_ms", "sd_batch1_median_latency_ms"),
        ("mean_throughput_mean_samples_per_second", "throughput_samples_per_second", "sd_throughput_mean_samples_per_second"),
        ("mean_parameter_overhead_pct_vs_A0", "parameter_overhead_pct_vs_A0", "sd_parameter_overhead_pct_vs_A0"),
        ("mean_peak_cuda_allocated_bytes", "peak_cuda_allocated_bytes", "sd_peak_cuda_allocated_bytes"),
    ]
    for record in summary.to_dict("records"):
        for source, metric, sd in measures:
            rows.append(_row(archives, "phase12", "phase12_efficiency_consolidated_summary.csv", system_id=record["system_id"], metric=metric, value=record[source], aggregation_unit="equal mean across 10 model seeds", limitation="Tesla T4 hardware cohort; timings are hardware-specific.", comparability_group="phase12_tesla_t4_efficiency", sample_sd=record[sd], gpu_name="Tesla T4"))
    return _frame(rows)


def build_locked_test_table(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows: list[dict] = []
    summary = read_csv_member(archives["phase14"], "phase14_equal_seed_summary.csv")
    for record in summary.to_dict("records"):
        rows.append(_row(archives, "phase14", "phase14_equal_seed_summary.csv", source_phase="14", system_id=record["system_id"], metric=f"{record['variant']}_{record['metric']}", value=record["mean"], aggregation_unit="equal mean across 10 frozen model seeds", limitation="One locked-test opening; repeated seeds share the same test rows.", comparability_group="locked_test_five_class", sample_sd=record["sd"], positive_seed_fraction=record["positive_fraction"]))
    return _frame(rows)


def build_external_validation_table(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows: list[dict] = []
    details = {
        "phase15a": ("15A", "sdcnl_pair_boundary", "Semantically aligned binary boundary; exploratory transfer only."),
        "phase15b": ("15B", "depressionreddit_proxy_boundary", "Others_proxy is not equivalent to the internal Others class."),
    }
    for phase_id, (phase_label, group, limitation) in details.items():
        summary = read_csv_member(archives[phase_id], "external_equal_seed_summary.csv")
        for record in summary.to_dict("records"):
            rows.append(_row(archives, phase_id, "external_equal_seed_summary.csv", source_phase=phase_label, system_id=record["system_id"], metric=record["metric"], value=record["mean"], aggregation_unit="equal mean across 10 frozen model seeds", limitation=limitation + " No consistent external specialist superiority.", comparability_group=group, mean=record["mean"], sample_sd=record["sd"], positive_seed_fraction=record["positive_fraction"]))
    return _frame(rows)


def build_all_paper_tables(archives: dict[str, VerifiedArchive]) -> dict[str, pd.DataFrame]:
    return {
        "paper_baseline_and_boundary_results.csv": build_baseline_boundary_table(archives),
        "paper_routing_controls_results.csv": build_routing_controls_table(archives),
        "paper_mechanism_robustness_results.csv": build_mechanism_robustness_table(archives),
        "paper_efficiency_results.csv": build_efficiency_table(archives),
        "paper_locked_test_results.csv": build_locked_test_table(archives),
        "paper_external_validation_results.csv": build_external_validation_table(archives),
    }


CLAIM_COLUMNS = [
    "claim_id", "claim_text", "evidence_tier", "support_status",
    "source_phase", "source_archive", "source_member", "source_fields",
    "derivation", "prohibited_overclaim", "mandatory_limitation",
]


def build_claims_evidence_matrix(archives: dict[str, VerifiedArchive]) -> pd.DataFrame:
    rows = [
        ("LOCKED_PRIMARY", "Phase 14 provides the primary frozen internal comparison.", "PRIMARY_LOCKED_TEST", "SUPPORTED", "14", "phase14", "phase14_equal_seed_summary.csv", "system_id,metric,mean,sd", "Direct equal-seed summary", "Do not imply independent test samples.", "The same locked-test rows are reused across ten frozen seeds."),
        ("ROUTING_FROZEN", "The routing threshold was frozen at tau=0.10 on validation before test opening.", "PROVENANCE_ONLY", "SUPPORTED", "9", "phase9", "phase9_selected_config.json", "selected_tau,test_accessed", "Direct frozen configuration", "Do not say tau was selected on the locked test.", "Threshold selection is development-only."),
        ("EXTERNAL_RESULT", "Both external tasks have negative mean specialist-minus-global Macro-F1 deltas with heterogeneous seeds.", "EXPLORATORY_EXTERNAL", "SUPPORTED", "15A/15B", "phase15a", "external_equal_seed_summary.csv", "metric,mean,positive_fraction", "Separate equal-seed summaries", "Do not pool the two external datasets.", "No consistent external specialist superiority."),
        ("UNIVERSAL_GENERALIZATION", "Universal domain generalization.", "EXPLORATORY_EXTERNAL", "PROHIBITED", "15A/15B", "phase15a", "external_claim_guardrails.csv", "directional_improvement_claim_supported", "Guardrail", "Universal generalization is unsupported.", "Two exploratory pair-boundary tasks cannot establish universal generalization."),
        ("CLINICAL_DEPLOYMENT", "Clinical validity, safety, diagnosis, or deployment readiness.", "PRIMARY_LOCKED_TEST", "PROHIBITED", "14", "phase14", "phase14_claim_guardrails.csv", "directional_improvement_claim_supported", "Guardrail", "Clinical/deployment claims are outside scope.", "This is a research classification evaluation, not clinical validation."),
        ("FULL_EXTERNAL_FIVE_CLASS", "Phase 15 is a full five-class external evaluation.", "EXPLORATORY_EXTERNAL", "PROHIBITED", "15A/15B", "phase15a", "external_label_space_shift.csv", "five_class_top1_outside_pair_fraction", "Task-definition guardrail", "Phase 15 evaluates pair boundaries only.", "External tasks are not full five-class evaluations."),
        ("OTHERS_PROXY_EQUIVALENCE", "Others_proxy is equivalent to the internal Others class.", "EXPLORATORY_EXTERNAL", "PROHIBITED", "15B", "phase15b", "external_label_space_shift.csv", "five_class_top1_outside_pair_fraction", "Semantic guardrail", "Proxy equivalence is unsupported.", "Others_proxy is not semantically identical to internal Others."),
        ("EXT_CONSISTENT_SUPERIORITY", "Specialists consistently outperform global models externally.", "EXPLORATORY_EXTERNAL", "PROHIBITED", "15A/15B", "phase15a", "external_equal_seed_summary.csv", "macro_f1_delta,positive_fraction", "Negative mean deltas and seed heterogeneity", "Do not claim consistent superiority.", "No consistent external specialist superiority."),
        ("LOCKED_TAU_SELECTION", "tau=0.10 was selected on the locked test.", "PRIMARY_LOCKED_TEST", "PROHIBITED", "14", "phase14", "phase14_protocol.json", "frozen_tau", "Protocol guardrail", "Locked-test selection is false.", "tau=0.10 was frozen on validation in Phase 9."),
        ("EQUIVALENCE", "No significant difference proves equivalence or non-inferiority.", "PRIMARY_LOCKED_TEST", "PROHIBITED", "14", "phase14", "phase14_mcnemar_holm.csv", "holm_adjusted_p", "Inference guardrail", "Non-significance is not equivalence.", "No equivalence or non-inferiority margin was tested."),
        ("CAUSAL_MECHANISM", "Lexical masking or routing establishes a causal mechanism.", "SECONDARY_INTERNAL", "PROHIBITED", "11", "phase11", "phase11_lexical_robustness_summary.csv", "variant,mean_macro_f1", "Observational robustness analysis", "Do not infer causality.", "Lexical and error analyses are descriptive/observational."),
    ]
    output = []
    for claim_id, text, tier, status, phase, archive_key, member, fields, derivation, overclaim, limitation in rows:
        output.append({
            "claim_id": claim_id, "claim_text": text, "evidence_tier": tier,
            "support_status": status, "source_phase": phase,
            "source_archive": archives[archive_key].spec.filename,
            "source_member": member, "source_fields": fields,
            "derivation": derivation, "prohibited_overclaim": overclaim,
            "mandatory_limitation": limitation,
        })
    return pd.DataFrame(output, columns=CLAIM_COLUMNS)


def build_limitations_markdown(claims: pd.DataFrame, tables: dict[str, pd.DataFrame]) -> str:
    prohibited = claims.loc[claims["support_status"] == "PROHIBITED", "mandatory_limitation"].drop_duplicates()
    bullets = "\n".join(f"- {item}" for item in prohibited)
    return "# Final limitations\n\n" + bullets + "\n\n- Development, locked-test, SDCNL, and Depression Reddit metrics remain separate and are never pooled.\n- Seed-level summaries are repeated-model analyses over shared evaluation rows, not independent datasets.\n"


def build_final_report_markdown(
    inventory: pd.DataFrame, gates: pd.DataFrame,
    tables: dict[str, pd.DataFrame], claims: pd.DataFrame,
) -> str:
    external = tables["paper_external_validation_results.csv"]
    deltas = external.loc[external["metric"] == "macro_f1_delta", ["source_phase", "mean", "positive_seed_fraction"]]
    locked = tables["paper_locked_test_results.csv"]
    primary = locked.loc[locked["metric"] == "original_macro_f1", ["system_id", "value", "sample_sd"]]
    return (
        "# Phase 16 final evidence report\n\n"
        f"All {len(inventory)} frozen archives passed integrity verification and all {len(gates)} registered upstream gates passed.\n\n"
        "## Primary locked-test evidence\n\n```text\n" + primary.to_string(index=False) + "\n```\n\n"
        "## Exploratory external evidence\n\n```text\n" + deltas.to_string(index=False) + "\n```\n\n"
        "No consistent external specialist superiority. "
        "Phase 15A and Phase 15B remain separate pair-boundary transfer tasks.\n\n"
        "## Claim discipline\n\n"
        f"The claims matrix contains {(claims['support_status'] == 'PROHIBITED').sum()} explicit prohibited overclaims.\n"
    )


import platform
import shutil
import sys
import tempfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

WORKSPACE = Path.cwd().resolve()
OUTPUT_DIR = WORKSPACE / 'phase16_final_evidence_results'
OUTPUT_ZIP = WORKSPACE / 'MentalDistress_Phase16_Final_Evidence_Results.zip'
FIGURE_DIR = OUTPUT_DIR / "figures"
if OUTPUT_DIR.exists():
    if OUTPUT_DIR.resolve().parent != WORKSPACE:
        raise RuntimeError("Refusing unsafe output-directory removal")
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir()
FIGURE_DIR.mkdir()

matplotlib.rcParams["svg.hashsalt"] = "mental-distress-phase16"
if any(name in sys.modules for name in ("torch", "transformers")):
    raise RuntimeError("Phase 16 must run without PyTorch or Transformers")
print("Phase 16 CPU workspace:", WORKSPACE)
print("Required frozen archives:", len(ARCHIVE_SPECS))

Phase 16 CPU workspace: /run/media/sha-weed/bc7edb4c-9668-4b1f-aaef-c5c2373f499b/MentalDistress v3
Required frozen archives: 12


## 1. Deterministic helper self-tests

Exercise path-safety, output-manifest, and byte-identical ZIP behavior before reading evidence.

In [2]:
with tempfile.TemporaryDirectory() as temporary:
    temp = Path(temporary)
    assert validate_member_names(["root/a.csv", "root/sub/b.json"])
    for unsafe in (["root/../escape.csv"], ["/absolute.csv"], [r"root\alias.csv"]):
        try:
            validate_member_names(unsafe)
        except ValueError:
            pass
        else:
            raise AssertionError(f"unsafe member accepted: {unsafe}")
    sample = temp / "sample"
    sample.mkdir()
    (sample / "a.csv").write_text("x\n1\n", encoding="utf-8")
    write_sha256_manifest(sample)
    verify_output_manifest(sample)
    first = deterministic_zip(sample, temp / "first.zip")
    second = deterministic_zip(sample, temp / "second.zip")
    assert first == second
print("PASS: Phase 16 helper self-tests")

PASS: Phase 16 helper self-tests


## 2. Archive discovery and integrity

Resolve exact filenames from the current directory, require frozen SHA-256 identities, validate CRC/path structure, and verify covering manifests.

In [3]:
verified_archives = resolve_required_archives(WORKSPACE)
archive_inventory = archive_inventory_frame(verified_archives)
assert len(verified_archives) == 12
assert archive_inventory["status"].eq("PASS").all()
print(archive_inventory[["phase_id", "filename", "member_count", "status"]].to_string(index=False))
print("PASS: twelve frozen ZIP archives verified")

phase_id                                                                 filename  member_count status
 step2_3                                 MentalDistress_Step2_Step3_Artifacts.zip            20   PASS
phase3_4         mentalroberta_phase3_freeze_phase4_boundary_analysis_RESULTS.zip            18   PASS
  phase5                    MentalDistress_Phase5_Bootstrap_Stability_Results.zip            13   PASS
  phase6                     MentalDistress_Phase6_Others_Sensitivity_Results.zip            19   PASS
  phase8                   MentalDistress_Phase8_All_Pair_Specialists_Results.zip           552   PASS
  phase9                    MentalDistress_Phase9_Routing_Calibration_Results.zip            34   PASS
 phase10               MentalDistress_Phase10_Controls_and_Allocation_Results.zip           226   PASS
 phase11      MentalDistress_Phase11_Mechanism_Reliability_Robustness_Results.zip            84   PASS
 phase12               MentalDistress_Phase12_Efficiency_Benchmarking_Res

## 3. Upstream phase gates

Check frozen seeds, H, K, tau, reproduction counts, audit status, and the one-time locked-test opening record.

In [4]:
archive_inventory, upstream_gate_audit = verify_all_phase_gates(verified_archives)
assert upstream_gate_audit["passed"].all()
print(upstream_gate_audit.groupby("phase_id")["passed"].agg(["count", "all"]).to_string())
print("PASS: every registered upstream phase gate")

          count   all
phase_id             
phase10       5  True
phase11       4  True
phase12       5  True
phase14       7  True
phase15a      7  True
phase15b      7  True
phase3_4      5  True
phase5        3  True
phase6        3  True
phase8        5  True
phase9        4  True
step2_3       7  True
PASS: every registered upstream phase gate


## 4. Evidence normalization

Build the experiment ledger and six provenance-rich paper tables without pooling incomparable scopes.

In [5]:
experiment_ledger = build_experiment_ledger(verified_archives)
paper_tables = build_all_paper_tables(verified_archives)
assert len(experiment_ledger) == 12
for name, frame in paper_tables.items():
    assert not frame.empty, name
    assert set(PROVENANCE_COLUMNS).issubset(frame.columns), name
    assert_no_raw_text_columns(frame)
print("Paper tables:")
for name, frame in paper_tables.items():
    print(f"  {name}: {len(frame)} rows")

Paper tables:
  paper_baseline_and_boundary_results.csv: 35 rows
  paper_routing_controls_results.csv: 16 rows
  paper_mechanism_robustness_results.csv: 20 rows
  paper_efficiency_results.csv: 15 rows
  paper_locked_test_results.csv: 33 rows
  paper_external_validation_results.csv: 14 rows


## 5. Claims and limitations

Create deterministic claim-to-evidence links and explicit prohibited overclaims, retaining null and negative external findings.

In [6]:
claims_evidence_matrix = build_claims_evidence_matrix(verified_archives)
final_limitations = build_limitations_markdown(claims_evidence_matrix, paper_tables)
final_results_report = build_final_report_markdown(
    archive_inventory, upstream_gate_audit, paper_tables, claims_evidence_matrix
)
assert (claims_evidence_matrix["support_status"] == "PROHIBITED").sum() >= 8
assert "No consistent external specialist superiority." in final_results_report
print(claims_evidence_matrix[["claim_id", "evidence_tier", "support_status"]].to_string(index=False))

                  claim_id        evidence_tier support_status
            LOCKED_PRIMARY  PRIMARY_LOCKED_TEST      SUPPORTED
            ROUTING_FROZEN      PROVENANCE_ONLY      SUPPORTED
           EXTERNAL_RESULT EXPLORATORY_EXTERNAL      SUPPORTED
  UNIVERSAL_GENERALIZATION EXPLORATORY_EXTERNAL     PROHIBITED
       CLINICAL_DEPLOYMENT  PRIMARY_LOCKED_TEST     PROHIBITED
  FULL_EXTERNAL_FIVE_CLASS EXPLORATORY_EXTERNAL     PROHIBITED
  OTHERS_PROXY_EQUIVALENCE EXPLORATORY_EXTERNAL     PROHIBITED
EXT_CONSISTENT_SUPERIORITY EXPLORATORY_EXTERNAL     PROHIBITED
      LOCKED_TAU_SELECTION  PRIMARY_LOCKED_TEST     PROHIBITED
               EQUIVALENCE  PRIMARY_LOCKED_TEST     PROHIBITED
          CAUSAL_MECHANISM   SECONDARY_INTERNAL     PROHIBITED


## 6. Publication figures

Write six PNG/SVG figure pairs with development, locked-test, and external scopes visually separated.

In [7]:
def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=180, bbox_inches="tight", metadata={"Software": "MentalDistress Phase 16"})
    fig.savefig(FIGURE_DIR / f"{stem}.svg", bbox_inches="tight", metadata={"Date": None})
    plt.close(fig)


# 1. Timeline and evidence hierarchy.
fig, ax = plt.subplots(figsize=(10, 4))
colors = {"PROVENANCE_ONLY": "#8da0cb", "SECONDARY_INTERNAL": "#66c2a5", "PRIMARY_LOCKED_TEST": "#fc8d62", "EXPLORATORY_EXTERNAL": "#e78ac3"}
ax.scatter(experiment_ledger["order"], np.zeros(len(experiment_ledger)), s=170, c=[colors[t] for t in experiment_ledger["evidence_tier"]])
for row in experiment_ledger.itertuples():
    ax.text(row.order, 0.05 if row.order % 2 else -0.05, row.phase_id, rotation=45, ha="center", va="bottom" if row.order % 2 else "top", fontsize=8)
ax.set(yticks=[], xlabel="Frozen experimental sequence", title="Evidence timeline and hierarchy")
save_figure(fig, "figure1_evidence_timeline")

# 2. Development controls and locked test remain in separate panels.
routing = paper_tables["paper_routing_controls_results.csv"]
locked = paper_tables["paper_locked_test_results.csv"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dev = routing[(routing["source_phase"] == "10") & (routing["metric"] == "macro_f1")]
axes[0].bar(dev["system_id"], dev["value"], color="#66c2a5")
axes[0].set(title="Development controls", ylabel="Macro-F1", ylim=(0.84, 0.88)); axes[0].tick_params(axis="x", rotation=45)
test = locked[locked["metric"] == "original_macro_f1"]
axes[1].bar(test["system_id"], test["value"], color="#fc8d62")
axes[1].set(title="Locked test (separate scope)", ylabel="Macro-F1", ylim=(0.82, 0.87)); axes[1].tick_params(axis="x", rotation=45)
save_figure(fig, "figure2_development_and_locked_test")

# 3. Routing coverage-performance curve on development validation only.
grid = routing[routing["source_phase"] == "9"].sort_values("tau")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(grid["coverage"], grid["value"], marker="o", color="#1b9e77")
selected = grid[grid["selected_tau"]]
ax.scatter(selected["coverage"], selected["value"], s=130, facecolors="none", edgecolors="#d95f02", linewidths=2)
ax.set(xlabel="Mean routing coverage", ylabel="Validation Macro-F1", title="Frozen routing calibration (tau=0.10 circled)")
save_figure(fig, "figure3_routing_coverage_performance")

# 4. Lexical robustness (descriptive, development only).
mechanism = paper_tables["paper_mechanism_robustness_results.csv"]
lexical = mechanism[mechanism["source_member"] == "phase11_lexical_robustness_summary.csv"]
fig, ax = plt.subplots(figsize=(8, 4))
pivot = lexical.pivot(index="system_id", columns="metric", values="value")
pivot.plot.bar(ax=ax, color=["#7570b3", "#e7298a", "#66a61e"][:len(pivot.columns)])
ax.set(title="Development lexical robustness (descriptive)", ylabel="Macro-F1", ylim=(0.75, 0.89)); ax.legend(fontsize=7)
save_figure(fig, "figure4_mechanism_robustness")

# 5. Efficiency trade-off on the Tesla T4 cohort.
efficiency = paper_tables["paper_efficiency_results.csv"]
latency = efficiency[efficiency["metric"] == "batch1_median_latency_ms"].set_index("system_id")
performance = efficiency[efficiency["metric"] == "macro_f1"].set_index("system_id")
fig, ax = plt.subplots(figsize=(6, 4))
for system in performance.index:
    ax.scatter(latency.loc[system, "value"], performance.loc[system, "value"], s=80, label=system)
ax.set(xlabel="Median batch-1 latency (ms)", ylabel="Development Macro-F1", title="Tesla T4 efficiency cohort"); ax.legend(fontsize=8)
save_figure(fig, "figure5_efficiency_tradeoff")

# 6. External tasks remain in separate panels.
external = paper_tables["paper_external_validation_results.csv"]
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, phase, title in zip(axes, ["15A", "15B"], ["SDCNL", "Depression Reddit proxy"]):
    row = external[(external["source_phase"] == phase) & (external["metric"] == "macro_f1_delta")].iloc[0]
    ax.bar(["specialist - global"], [row["mean"]], color="#e78ac3")
    ax.axhline(0, color="black", linewidth=1)
    ax.set(title=title, ylabel="Mean Macro-F1 delta")
save_figure(fig, "figure6_external_deltas_separate")
print("PASS: six publication figure pairs created")

PASS: six publication figure pairs created


## 7. Protocol, environment, and execution summary

Record the frozen reporting protocol and CPU environment.

In [8]:
protocol = {
    "phase": 16,
    "title": "Final Evidence Consolidation",
    "execution_device": "CPU",
    "input_policy": "exact twelve local frozen ZIP archives",
    "archive_order": list(ARCHIVE_ORDER),
    "evidence_tiers": {key: spec.evidence_tier for key, spec in ARCHIVE_SPECS.items()},
    "frozen_H": list(HARD_PAIRS),
    "frozen_K": 3,
    "frozen_tau": 0.10,
    "raw_datasets_opened": False,
    "models_loaded": False,
    "inference_performed": False,
    "metrics_pooled_across_scopes": False,
}
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_device": "CPU",
}
execution_summary = {
    "phase": 16,
    "status": "PASS",
    "verified_archive_count": len(verified_archives),
    "upstream_gate_count": len(upstream_gate_audit),
    "failed_gate_count": int((~upstream_gate_audit["passed"]).sum()),
    "paper_table_count": len(paper_tables),
    "figure_pair_count": 6,
    "prohibited_claim_count": int((claims_evidence_matrix["support_status"] == "PROHIBITED").sum()),
    "raw_datasets_opened": False,
    "models_loaded": False,
    "test_reopened": False,
}
print(json.dumps(execution_summary, indent=2, sort_keys=True))

{
  "failed_gate_count": 0,
  "figure_pair_count": 6,
  "models_loaded": false,
  "paper_table_count": 6,
  "phase": 16,
  "prohibited_claim_count": 8,
  "raw_datasets_opened": false,
  "status": "PASS",
  "test_reopened": false,
  "upstream_gate_count": 62,
  "verified_archive_count": 12
}


## 8. Output construction and structure audit

Write only consolidated summaries, reports, and figures—never upstream row-level predictions or raw text.

In [9]:
write_frame(OUTPUT_DIR / "phase16_input_archive_inventory.csv", archive_inventory)
write_frame(OUTPUT_DIR / "phase16_upstream_gate_audit.csv", upstream_gate_audit)
write_frame(OUTPUT_DIR / "phase16_experiment_ledger.csv", experiment_ledger)
for filename, frame in paper_tables.items():
    write_frame(OUTPUT_DIR / filename, frame)
write_frame(OUTPUT_DIR / "claims_evidence_matrix.csv", claims_evidence_matrix)
(OUTPUT_DIR / "final_limitations.md").write_text(final_limitations, encoding="utf-8", newline="\n")
(OUTPUT_DIR / "final_results_report.md").write_text(final_results_report, encoding="utf-8", newline="\n")
write_json(OUTPUT_DIR / "phase16_protocol.json", protocol)
(OUTPUT_DIR / "phase16_protocol.sha256").write_text(sha256_file(OUTPUT_DIR / "phase16_protocol.json") + "  phase16_protocol.json\n", encoding="utf-8", newline="\n")
write_json(OUTPUT_DIR / "phase16_environment.json", environment)
write_json(OUTPUT_DIR / "phase16_execution_summary.json", execution_summary)

expected_files = [
    "phase16_input_archive_inventory.csv", "phase16_upstream_gate_audit.csv",
    "phase16_experiment_ledger.csv", *paper_tables.keys(), "claims_evidence_matrix.csv",
    "final_limitations.md", "final_results_report.md", "phase16_protocol.json",
    "phase16_protocol.sha256", "phase16_environment.json", "phase16_execution_summary.json",
]
expected_files += [f"figures/figure{index}_{stem}.{extension}" for index, stem in [
    (1, "evidence_timeline"), (2, "development_and_locked_test"),
    (3, "routing_coverage_performance"), (4, "mechanism_robustness"),
    (5, "efficiency_tradeoff"), (6, "external_deltas_separate"),
] for extension in ("png", "svg")]
structure_rows = []
for relative in expected_files:
    path = OUTPUT_DIR / relative
    structure_rows.append({"check": f"exists:{relative}", "observed": path.is_file(), "expected": True, "passed": path.is_file()})
for filename, frame in paper_tables.items():
    no_raw = True
    try:
        assert_no_raw_text_columns(frame)
    except ValueError:
        no_raw = False
    structure_rows.append({"check": f"no_raw_text_columns:{filename}", "observed": no_raw, "expected": True, "passed": no_raw})
structure_audit = pd.DataFrame(structure_rows)
assert structure_audit["passed"].all()
write_frame(OUTPUT_DIR / "phase16_structure_audit.csv", structure_audit)
print("PASS: consolidated evidence outputs and structure audit written")

PASS: consolidated evidence outputs and structure audit written


## 9. Manifest, deterministic package, and completion

Verify the output manifest and require two independently built ZIP files to be byte-identical.

In [10]:
write_sha256_manifest(OUTPUT_DIR)
verify_output_manifest(OUTPUT_DIR)
first_sha = deterministic_zip(OUTPUT_DIR, OUTPUT_ZIP)
temporary_zip = WORKSPACE / ".phase16_determinism_check.zip"
second_sha = deterministic_zip(OUTPUT_DIR, temporary_zip)
assert first_sha == second_sha
assert OUTPUT_ZIP.read_bytes() == temporary_zip.read_bytes()
temporary_zip.unlink()
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    assert archive.testzip() is None
    assert len(archive.namelist()) == len(set(archive.namelist()))
print("PHASE 16 COMPLETE")
print("Result directory:", OUTPUT_DIR)
print("Result archive:", OUTPUT_ZIP)
print("Result archive SHA-256:", first_sha)

PHASE 16 COMPLETE
Result directory: /run/media/sha-weed/bc7edb4c-9668-4b1f-aaef-c5c2373f499b/MentalDistress v3/phase16_final_evidence_results
Result archive: /run/media/sha-weed/bc7edb4c-9668-4b1f-aaef-c5c2373f499b/MentalDistress v3/MentalDistress_Phase16_Final_Evidence_Results.zip
Result archive SHA-256: 2f7ea9ffec9eab2bd83e2e11dbfa9d1480fa6b7a7a0e0f61f51a3026ef65e0ff
